In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

data = [
    (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df = spark.createDataFrame(data, columns)
df.show()



+--------+-----------+----------+------+------------+
|order_id|customer_id|   product|amount|updated_date|
+--------+-----------+----------+------+------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|
|       2|       C002|    Mobile|  NULL|  2024-01-02|
|       3|       C003|    Tablet| 20000|  2024-01-03|
|       4|       C004|    Laptop| 55000|  2024-01-04|
|       5|       C005|Headphones|  NULL|  2024-01-05|
|       6|       C006|    Camera| 30000|  2024-01-06|
|       7|       C007|    Mobile| 18000|  2024-01-07|
|       8|       C008|     Watch|  8000|  2024-01-07|
+--------+-----------+----------+------+------------+



In [0]:
from pyspark.sql.functions import col,when,lit
df=df.withColumn("amount",when(col("amount").isNull(),lit(1000)).otherwise(col("amount")))
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
from pyspark.sql.functions import to_date
df=df.withColumn("updated_date",to_date(col("updated_date"),"yyyy-MM-dd"))
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.withColumnRenamed("updated_date","date")
df.display()

order_id,customer_id,product,amount,date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.withColumn("bonus",col("amount")*0.1)\
    .withColumn("category",
                when(col("amount")>20000,"High").otherwise("Low"))
df.display()

order_id,customer_id,product,amount,date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,High
2,C002,Mobile,1000,2024-01-02,100.0,Low
3,C003,Tablet,20000,2024-01-03,2000.0,Low
4,C004,Laptop,55000,2024-01-04,5500.0,High
5,C005,Headphones,1000,2024-01-05,100.0,Low
6,C006,Camera,30000,2024-01-06,3000.0,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low
8,C008,Watch,8000,2024-01-07,800.0,Low


In [0]:
from pyspark.sql.types import *
def amount_bucket(amount):
    if amount<10000:
        return "Low"
    elif amount>10000 and amount <30000:
        return "Medium"
    elif amount==30000:
        return "High"
reg_amo_buc=udf(amount_bucket,StringType())
df.withColumn("amount_buck",reg_amo_buc(col("amount"))).display()

order_id,customer_id,product,amount,date,bonus,category,amount_buck
1,C001,Laptop,50000,2024-01-01,5000.0,High,null
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,Low,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,null
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,High,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/sample_data/orders_data")

In [0]:
Data=[
    (3, "C003", "Tablet", "22000", "2024-01-06"),
    (9, "C009", "Laptop", "60000", "2024-01-08"),
(10, "C010", "Tablet", None, "2024-01-08"),
(11, "C011", "Mobile", "15000", "2024-01-09")
]
columns=["order_id","customer_id","product","amount","date"]
df_new=spark.createDataFrame(Data,columns)



In [0]:
new_data=df.join(df_new,on="order_id",how="left_anti")
new_data.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
updated=new_data.union(df_new)
updated.orderBy("order_id").display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,22000,2024-01-06
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07
9,C009,Laptop,60000,2024-01-08
10,C010,Tablet,null,2024-01-08


In [0]:
dbutils.widgets.text("Text_Widget","text","Enter anything")



In [0]:
dbutils.widgets.dropdown(
    "DropDown",
    "Apple",
    ["Apple","Banana","Sapota"],
    "Choose one"
)

In [0]:
dbutils.widgets.combobox(
    "Combobox",
    "Apple",
    ["Apple","Banana","Sapota"],
    "Choose one"
)

In [0]:
dbutils.widgets.multiselect(
    "MultiSelect",
    "Apple",
    ["Apple","Banana","Sapota"],
    "Choose one"
)